# Trees cleaning


## 1  Setup

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np


# define which columns we want to keep
TREE_KEEP_COLS = [
    "CoM ID",
    "Common Name",
    "Scientific Name",
    "Genus",
    "Family",
    "Diameter Breast Height",
    "Year Planted",
    "Date Planted",
    "Age Description",
    "Useful Life Expectency",
    "Useful Life Expectency Value",
    "Precinct",
    "Located in",
    "Latitude",
    "Longitude",
]

# define rename mapping
TREE_RENAME_MAP = {
    "CoM ID": "com_id",
    "Common Name": "common_name",
    "Scientific Name": "scientific_name",
    "Genus": "genus",
    "Family": "family",
    "Diameter Breast Height": "diameter_breast_height",
    "Year Planted": "year_planted",
    "Date Planted": "date_planted",
    "Age Description": "age_description",
    "Useful Life Expectency": "useful_life_expectency",
    "Useful Life Expectency Value": "useful_life_expectency_value",
    "Precinct": "precinct",
    "Located in": "located_in",
    "Latitude": "latitude",
    "Longitude": "longitude",
}



## 2 — Load 


In [ ]:
def load_tree_data(filepath: Path) -> pd.DataFrame:
    """
    Read raw tree CSV, keep only needed columns, and rename them to snake_case.
    """
    # Read raw file
    df = pd.read_csv(filepath, encoding="utf-8-sig")

    # Print raw shape and raw columns for checking
    print("Raw shape:", df.shape)
    print("\nRaw columns:")
    print(df.columns.tolist())

    # Keep only selected columns
    df = df[TREE_KEEP_COLS].copy()

    # Rename to snake_case
    df = df.rename(columns=TREE_RENAME_MAP)

    # Print result for checking
    print("\nAfter Step 1 shape:", df.shape)
    print("\nAfter Step 1 columns:")
    print(df.columns.tolist())

    return df

# load the data
tree_path = Path("trees-with-species-and-dimensions-urban-forest.csv")

tree_raw = load_tree_data(tree_path)

tree_raw.head()


Raw shape: (82064, 20)

Raw columns:
['CoM ID', 'Common Name', 'Scientific Name', 'Genus', 'Family', 'Diameter Breast Height', 'Year Planted', 'Date Planted', 'Age Description', 'Useful Life Expectency', 'Useful Life Expectency Value', 'Precinct', 'Located in', 'UploadDate', 'CoordinateLocation', 'Latitude', 'Longitude', 'Easting', 'Northing', 'geolocation']

After Step 1 shape: (82064, 15)

After Step 1 columns:
['com_id', 'common_name', 'scientific_name', 'genus', 'family', 'diameter_breast_height', 'year_planted', 'date_planted', 'age_description', 'useful_life_expectency', 'useful_life_expectency_value', 'precinct', 'located_in', 'latitude', 'longitude']


,com_id,common_name,scientific_name,genus,family,diameter_breast_height,year_planted,date_planted,age_description,useful_life_expectency,useful_life_expectency_value,precinct,located_in,latitude,longitude
0,1070378,Tulip Tree,Liriodendron tulipifera,Liriodendron,Magnoliaceae,20.0,2006,2006-12-15,Mature,> 41 years,50,South Yarra,Street,-37.832567,144.986879
1,1070382,Tulip Tree,Liriodendron tulipifera,Liriodendron,Magnoliaceae,21.0,2006,2006-12-15,Mature,> 41 years,50,South Yarra,Street,-37.831669,144.987059
2,1796650,Cook pine,Araucaria columnaris,Araucaria,Araucariaceae,NaN,2020,2020-12-14,Semi-mature,21 - 30 years,30,Carlton,Park,-37.802222,144.962852
3,1457913,Yellow Box,Eucalyptus melliodora,Eucalyptus,Myrtaceae,25.0,2010,2010-12-14,Mature,> 41 years,50,Kensington,Park,-37.797537,144.923519
4,1457915,Yellow Box,Eucalyptus melliodora,Eucalyptus,Myrtaceae,22.0,2010,2010-12-14,Mature,> 41 years,50,Kensington,Park,-37.797540,144.923459


## 3 clean


In [ ]:
TEXT_COLS = [
    "common_name",
    "scientific_name",
    "genus",
    "family",
    "age_description",
    "useful_life_expectency",
    "precinct",
    "located_in",
]

# Melbourne range for validation
LAT_MIN, LAT_MAX = -38.1, -37.5
LON_MIN, LON_MAX = 144.7, 145.3

def clean_tree_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic cleaning for tree data:
    1. fill missing common_name with scientific_name
    2. drop rows missing core task fields
    3. trim text columns
    4. repair encoding artifacts
    5. convert dtypes
    6. validate coordinate range
    7. deduplicate by com_id
    """
    clean_df = df.copy()

    # fill missing common_name using scientific_name
    clean_df["common_name"] = clean_df["common_name"].fillna(clean_df["scientific_name"])

    # drop rows missing core task fields
    # For later tree task generation, we need at least name + coordinates
    clean_df = clean_df.dropna(subset=["common_name", "latitude", "longitude"])

    # trim text columns
    for col in TEXT_COLS:
        clean_df[col] = clean_df[col].astype("string").str.strip()

    # repair encoding artifacts
    # Example: HillŸ??s Weeping Fig -> Hill's Weeping Fig
    for col in TEXT_COLS:
        clean_df[col] = (
            clean_df[col]
            .str.replace("Ÿ??", "'", regex=False)
            .str.replace("??", "", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    # convert dtypes
    clean_df["com_id"] = pd.to_numeric(clean_df["com_id"], errors="coerce").astype("Int64")
    clean_df["diameter_breast_height"] = pd.to_numeric(
        clean_df["diameter_breast_height"], errors="coerce"
    )
    clean_df["year_planted"] = pd.to_numeric(
        clean_df["year_planted"], errors="coerce"
    ).astype("Int64")
    clean_df["useful_life_expectency_value"] = pd.to_numeric(
        clean_df["useful_life_expectency_value"], errors="coerce"
    )
    clean_df["latitude"] = pd.to_numeric(clean_df["latitude"], errors="coerce")
    clean_df["longitude"] = pd.to_numeric(clean_df["longitude"], errors="coerce")
    clean_df["date_planted"] = pd.to_datetime(clean_df["date_planted"], errors="coerce")

    # coordinate validation
    clean_df = clean_df[
        clean_df["latitude"].between(LAT_MIN, LAT_MAX)
        & clean_df["longitude"].between(LON_MIN, LON_MAX)
    ].copy()

    # remove rows with invalid com_id, then deduplicate
    clean_df = clean_df.dropna(subset=["com_id"])
    clean_df = clean_df.drop_duplicates(subset=["com_id"], keep="first").copy()

    # convert empty-like strings back to missing
    for col in TEXT_COLS:
        clean_df[col] = clean_df[col].replace({"": pd.NA, "nan": pd.NA, "<NA>": pd.NA})

    return clean_df

tree_clean = clean_tree_data(tree_raw)

print("tree_raw shape :", tree_raw.shape)
print("tree_clean shape:", tree_clean.shape)

tree_clean.head()




tree_raw shape : (82064, 15)
tree_clean shape: (82064, 15)


,com_id,common_name,scientific_name,genus,family,diameter_breast_height,year_planted,date_planted,age_description,useful_life_expectency,useful_life_expectency_value,precinct,located_in,latitude,longitude
0,1070378,Tulip Tree,Liriodendron tulipifera,Liriodendron,Magnoliaceae,20.0,2006,2006-12-15,Mature,> 41 years,50,South Yarra,Street,-37.832567,144.986879
1,1070382,Tulip Tree,Liriodendron tulipifera,Liriodendron,Magnoliaceae,21.0,2006,2006-12-15,Mature,> 41 years,50,South Yarra,Street,-37.831669,144.987059
2,1796650,Cook pine,Araucaria columnaris,Araucaria,Araucariaceae,NaN,2020,2020-12-14,Semi-mature,21 - 30 years,30,Carlton,Park,-37.802222,144.962852
3,1457913,Yellow Box,Eucalyptus melliodora,Eucalyptus,Myrtaceae,25.0,2010,2010-12-14,Mature,> 41 years,50,Kensington,Park,-37.797537,144.923519
4,1457915,Yellow Box,Eucalyptus melliodora,Eucalyptus,Myrtaceae,22.0,2010,2010-12-14,Mature,> 41 years,50,Kensington,Park,-37.797540,144.923459



Missing values after Step 2:
diameter_breast_height    44863
useful_life_expectency    17410
com_id                        0
common_name                   0
scientific_name               0
genus                         0
family                        0
year_planted                  0
date_planted                  0
age_description               0
dtype: int64

Duplicate com_id after Step 2:
0

Coordinate range after Step 2:
           latitude     longitude
count  82064.000000  82064.000000
mean     -37.801873    144.950772
std        0.016644      0.019349
min      -37.850518    144.900370
25%      -37.815692    144.941931
50%      -37.797766    144.951071
75%      -37.789021    144.962702
max      -37.775511    144.991056


## 4  Generate location_group and location_label for the tree.




In [7]:
print("Missing precinct:", tree_clean["precinct"].isna().sum())
print("Missing located_in:", tree_clean["located_in"].isna().sum())

print("\nTop 20 precinct values:")
print(tree_clean["precinct"].value_counts(dropna=False).head(20))

print("\nTop 20 located_in values:")
print(tree_clean["located_in"].value_counts(dropna=False).head(20))

Missing precinct: 0
Missing located_in: 0

Top 20 precinct values:
precinct
Parkville                   28567
Melbourne                    9063
Kensington                   7555
West Melbourne               6756
Docklands                    5816
North Melbourne              4715
East Melbourne               4506
Carlton                      4368
South Yarra                  2631
Port Melbourne               2623
Southbank                    2338
Carlton North                2323
Flemington                    510
North And West Melbourne      113
Princes Hill                   65
Fishermans Bend                61
Central City                   26
Brunswick West                 24
Richmond                        2
Brunswick                       2
Name: count, dtype: Int64

Top 20 located_in values:
located_in
Park      48262
Street    33802
Name: count, dtype: Int64


In [9]:
def normalize_place_text(value):
    """
    Normalize a place-related text value for location grouping.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if value == "":
        return pd.NA

    # collapse repeated spaces
    value = " ".join(value.split())

    return value


def build_tree_location_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build location_group and location_label for tree data.
    """
    tree_df_cleaned = df.copy()

    tree_df_cleaned["precinct_norm"] = tree_df_cleaned["precinct"].apply(normalize_place_text)
    tree_df_cleaned["located_in_norm"] = tree_df_cleaned["located_in"].apply(normalize_place_text)

    def combine_location(row):
        precinct = row["precinct_norm"]
        located_in = row["located_in_norm"]

        if pd.notna(precinct) and pd.notna(located_in):
            return f"{precinct} | {located_in}"
        elif pd.notna(precinct):
            return precinct
        elif pd.notna(located_in):
            return located_in
        else:
            return pd.NA

    tree_df_cleaned["location_group"] = tree_df_cleaned.apply(combine_location, axis=1)
    tree_df_cleaned["location_label"] = tree_df_cleaned["location_group"]

    return tree_df_cleaned



In [11]:
tree_loc = build_tree_location_fields(tree_clean)

print(tree_loc[["precinct", "located_in", "location_group", "location_label"]].head(10))

print("Missing location_group:", tree_loc["location_group"].isna().sum())
print("Unique location_group:", tree_loc["location_group"].nunique(dropna=True))

print("\nTop 30 location_group values:")
print(tree_loc["location_group"].value_counts(dropna=False).head(30))

      precinct located_in        location_group        location_label
0  South Yarra     Street  South Yarra | Street  South Yarra | Street
1  South Yarra     Street  South Yarra | Street  South Yarra | Street
2      Carlton       Park        Carlton | Park        Carlton | Park
3   Kensington       Park     Kensington | Park     Kensington | Park
4   Kensington       Park     Kensington | Park     Kensington | Park
5   Kensington       Park     Kensington | Park     Kensington | Park
6   Kensington       Park     Kensington | Park     Kensington | Park
7   Kensington       Park     Kensington | Park     Kensington | Park
8   Kensington       Park     Kensington | Park     Kensington | Park
9   Kensington       Park     Kensington | Park     Kensington | Park
Missing location_group: 0
Unique location_group: 37

Top 30 location_group values:
location_group
Parkville | Park                     24853
Melbourne | Park                      4978
Melbourne | Street                    4085
Nor

## 5 tree task build

In [15]:
def build_tree_task_content(
    df: pd.DataFrame,
    series_id: int = 1,
    geofence_radius_meter: int = 50,
    base_difficulty: int = 1,
    reward_point: int = 10,
) -> pd.DataFrame:
    """
    Build object-based task content for tree data.
    Use short description + short criteria style to align with backend sample.
    """
    output_df = df.copy()

    def clean_species_name(x):
        if pd.isna(x):
            return pd.NA
        return str(x).strip()

    # Clean common name for task text generation
    output_df["common_name_clean"] = output_df["common_name"].apply(clean_species_name)

    # build  task name
    output_df["task_name"] = output_df["common_name_clean"]

    # task_description: short user-facing instruction with location hint
    output_df["task_description"] = output_df.apply(
        lambda row: f"Take a picture of a {row['common_name_clean']} near {row['location_label']}.",
        axis=1
    )

    # evaluation_criteria short and judgeable for AI
    output_df["evaluation_criteria"] = output_df["common_name_clean"].apply(
        lambda x: f"A {x} tree should be present in the image."
    )

    # Default task fields
    output_df["series_id"] = series_id
    output_df["environment_type"] = "outdoor"
    output_df["task_type"] = "photo"
    output_df["geofence_radius_meter"] = geofence_radius_meter
    output_df["base_difficulty"] = base_difficulty
    output_df["reward_point"] = reward_point
    output_df["is_active"] = True

    return output_df



In [16]:
# tree tasks intialization
tree_task_base = build_tree_task_content(tree_loc)

tree_task_base.head(10)

,com_id,common_name,scientific_name,genus,family,diameter_breast_height,year_planted,date_planted,age_description,useful_life_expectency,...,task_name,task_description,evaluation_criteria,series_id,environment_type,task_type,geofence_radius_meter,base_difficulty,reward_point,is_active
0,1070378,Tulip Tree,Liriodendron tulipifera,Liriodendron,Magnoliaceae,20.0,2006,2006-12-15,Mature,> 41 years,...,Tulip Tree,Take a picture of a Tulip Tree near South Yarr...,A Tulip Tree tree should be present in the image.,1,outdoor,photo,50,1,10,True
1,1070382,Tulip Tree,Liriodendron tulipifera,Liriodendron,Magnoliaceae,21.0,2006,2006-12-15,Mature,> 41 years,...,Tulip Tree,Take a picture of a Tulip Tree near South Yarr...,A Tulip Tree tree should be present in the image.,1,outdoor,photo,50,1,10,True
2,1796650,Cook pine,Araucaria columnaris,Araucaria,Araucariaceae,NaN,2020,2020-12-14,Semi-mature,21 - 30 years,...,Cook pine,Take a picture of a Cook pine near Carlton | P...,A Cook pine tree should be present in the image.,1,outdoor,photo,50,1,10,True
3,1457913,Yellow Box,Eucalyptus melliodora,Eucalyptus,Myrtaceae,25.0,2010,2010-12-14,Mature,> 41 years,...,Yellow Box,Take a picture of a Yellow Box near Kensington...,A Yellow Box tree should be present in the image.,1,outdoor,photo,50,1,10,True
4,1457915,Yellow Box,Eucalyptus melliodora,Eucalyptus,Myrtaceae,22.0,2010,2010-12-14,Mature,> 41 years,...,Yellow Box,Take a picture of a Yellow Box near Kensington...,A Yellow Box tree should be present in the image.,1,outdoor,photo,50,1,10,True
5,1457917,Flame Tree,Brachychiton acerifolius,Brachychiton,Malvaceae,11.0,2010,2010-12-14,Mature,> 41 years,...,Flame Tree,Take a picture of a Flame Tree near Kensington...,A Flame Tree tree should be present in the image.,1,outdoor,photo,50,1,10,True
6,1457918,Brittle Gum,Eucalyptus mannifera,Eucalyptus,Myrtaceae,24.0,2010,2010-12-14,Mature,> 41 years,...,Brittle Gum,Take a picture of a Brittle Gum near Kensingto...,A Brittle Gum tree should be present in the im...,1,outdoor,photo,50,1,10,True
7,1457929,Yellow Box,Eucalyptus melliodora,Eucalyptus,Myrtaceae,80.0,2010,2010-12-14,Mature,> 41 years,...,Yellow Box,Take a picture of a Yellow Box near Kensington...,A Yellow Box tree should be present in the image.,1,outdoor,photo,50,1,10,True
8,1457931,River red gum,Eucalyptus camaldulensis,Eucalyptus,Myrtaceae,20.0,2010,2010-12-14,Mature,> 41 years,...,River red gum,Take a picture of a River red gum near Kensing...,A River red gum tree should be present in the ...,1,outdoor,photo,50,1,10,True
9,1457935,Black She Oak,Allocasuarina littoralis,Allocasuarina,Casuarinaceae,80.0,2010,2010-12-14,Mature,21 - 30 years,...,Black She Oak,Take a picture of a Black She Oak near Kensing...,A Black She Oak tree should be present in the ...,1,outdoor,photo,50,1,10,True


## 6 Determination of final cols


In [ ]:
FINAL_COLS = [
    "series_id",
    "com_id",
    "task_name",
    "task_description",
    "evaluation_criteria",
    "environment_type",
    "task_type",
    "location_label",
    "latitude",
    "longitude",
    "st_point",
    "geofence_radius_meter",
    "base_difficulty",
    "reward_point",
    "is_active",
]

def build_st_point(longitude, latitude):
 
    if pd.isna(longitude) or pd.isna(latitude):
        return pd.NA
    return f"POINT({longitude} {latitude})"


def build_tree_task_seed(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build final tree task seed aligned to the agreed export schema
    """
    output_col = df.copy()

    # build SQL-ready point text
    output_col["st_point"] = output_col.apply(
        lambda row: build_st_point(row["longitude"], row["latitude"]),
        axis=1
    )

    # keep only final export columns
    output_col = output_col[FINAL_COLS].copy()

    return output_col

tree_task_seed = build_tree_task_seed(tree_task_base)

print(tree_task_seed.shape)
tree_task_seed.head(10)



(82064, 15)


,series_id,com_id,task_name,task_description,evaluation_criteria,environment_type,task_type,location_label,latitude,longitude,st_point,geofence_radius_meter,base_difficulty,reward_point,is_active
0,1,1070378,Tulip Tree,Take a picture of a Tulip Tree near South Yarr...,A Tulip Tree tree should be present in the image.,outdoor,photo,South Yarra | Street,-37.832567,144.986879,POINT(144.986879 -37.83256738),50,1,10,True
1,1,1070382,Tulip Tree,Take a picture of a Tulip Tree near South Yarr...,A Tulip Tree tree should be present in the image.,outdoor,photo,South Yarra | Street,-37.831669,144.987059,POINT(144.9870586 -37.83166873),50,1,10,True
2,1,1796650,Cook pine,Take a picture of a Cook pine near Carlton | P...,A Cook pine tree should be present in the image.,outdoor,photo,Carlton | Park,-37.802222,144.962852,POINT(144.9628525 -37.80222191),50,1,10,True
3,1,1457913,Yellow Box,Take a picture of a Yellow Box near Kensington...,A Yellow Box tree should be present in the image.,outdoor,photo,Kensington | Park,-37.797537,144.923519,POINT(144.9235189 -37.79753724),50,1,10,True
4,1,1457915,Yellow Box,Take a picture of a Yellow Box near Kensington...,A Yellow Box tree should be present in the image.,outdoor,photo,Kensington | Park,-37.797540,144.923459,POINT(144.9234592 -37.79753979),50,1,10,True
5,1,1457917,Flame Tree,Take a picture of a Flame Tree near Kensington...,A Flame Tree tree should be present in the image.,outdoor,photo,Kensington | Park,-37.797437,144.923580,POINT(144.9235797 -37.79743737),50,1,10,True
6,1,1457918,Brittle Gum,Take a picture of a Brittle Gum near Kensingto...,A Brittle Gum tree should be present in the im...,outdoor,photo,Kensington | Park,-37.797480,144.923442,POINT(144.9234421 -37.79747994),50,1,10,True
7,1,1457929,Yellow Box,Take a picture of a Yellow Box near Kensington...,A Yellow Box tree should be present in the image.,outdoor,photo,Kensington | Park,-37.797282,144.922884,POINT(144.9228843 -37.79728161),50,1,10,True
8,1,1457931,River red gum,Take a picture of a River red gum near Kensing...,A River red gum tree should be present in the ...,outdoor,photo,Kensington | Park,-37.797315,144.922794,POINT(144.9227939 -37.79731543),50,1,10,True
9,1,1457935,Black She Oak,Take a picture of a Black She Oak near Kensing...,A Black She Oak tree should be present in the ...,outdoor,photo,Kensington | Park,-37.797244,144.922634,POINT(144.9226343 -37.7972436),50,1,10,True


## 7 — Export `trees_cleaned.csv`


In [19]:
output_path = Path("task_seed_tree_nature.csv")

tree_task_seed.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Exported to: {output_path}")
print(f"Row count: {len(tree_task_seed)}")
print(f"Column count: {len(tree_task_seed.columns)}")

Exported to: task_seed_tree_nature.csv
Row count: 82064
Column count: 15
